# Pair 6: Two vision-language models answering questions about product-style photos

**Brief reference:** Section 2.1 (multimodal learning, large multimodal models), Section 2.2 (objectives, tasks, functions, framework, algorithms, governance, advantages), Section 3.1 to 3.3.

**Scenario.** A home-decor retailer wants a model that answers shopper and support-staff questions about photos (what is in the picture, how many items, what colour, is an object present). Two recent open vision-language models are compared on identical questions, and the previous generation of the first one is kept as a reference:

| | System A | System B | Reference (not a system of this pair) |
|---|---|---|---|
| Model | Qwen3-VL-8B-Instruct (4-bit GGUF build, Ollama, October 2025) | InternVL3-8B (4-bit NF4 bitsandbytes, Hugging Face transformers, April 2025) | Qwen2.5-VL-7B-Instruct (4-bit GGUF, Ollama) |
| Language model | Qwen3 8B | Qwen2.5 7B inside InternVL3-8B | Qwen2.5 7B |
| Vision encoder | Qwen3-VL ViT with dynamic resolution | InternViT-300M with dynamic tiling | Qwen ViT with dynamic resolution |

Qwen3-VL-8B is run as its instruct variant (no reasoning mode). A first attempt with the default reasoning variant is kept as evidence in `results/feasibility/qwen3_reasoning_variant/`: it wrote its reasoning as text, used up the short answer budget on the open VQA questions and answered none of them. The reference model is also System B of pair 4 (the vision agent). It is included here so that the step from one Qwen generation to the next can be read directly, and so that every system evaluated in the assignment is a distinct model. The systems are close in size and all run 4-bit. They differ in vision encoder, training data, quantisation scheme and runtime, and the results must be read with that in mind (see limits at the end).

**Evaluation design.**
- **Hallucination (POPE, adversarial split):** yes/no questions about whether an object is present. Metrics: accuracy, precision, recall, F1, yes-ratio, hallucination rate (answering yes when the object is absent).
- **Visual question answering (VQAv2 validation, home and furniture subset):** questions selected by keyword, 100 per answer type (yes/no, number, other). Metric: VQA accuracy against the ten human answers.
- **Robustness:** the same POPE question is re-asked with different wording. The flip rate measures how often the answer changes.
- **Cost:** latency, VRAM, GPU energy and an assumption-based electricity cost.
- Temperature 0 and fixed sampling seeds. Random sampling of questions uses seed 0.
- GPU headroom matters on this laptop. A test with artificial memory ballast showed that roughly 7.8 GB is usable before the driver spills into shared memory. InternVL with a bf16 output layer used 7.6 GB and became 20 to 50 times slower inside the notebook. Quantising the output layer as well (6.7 GB, answers in about 1.7 s even with 600 MB of ballast) removed the problem.
- InternVL tiling is capped at 6 tiles plus a thumbnail (about 1,800 image tokens). Its default of 12 tiles (about 3,300 tokens) overflowed the 8 GB GPU in testing and made every answer 30 to 80 times slower. Qwen2.5-VL uses about 1,100 image tokens for the same photos.

In [ ]:
import os, sys, io, json, time, re, random, pathlib, platform, subprocess, threading, importlib.metadata as im, warnings, string
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from PIL import Image
import torch, ollama
warnings.filterwarnings("ignore")

QUICK = os.environ.get("P6_QUICK", "0") == "1"
HERE = pathlib.Path.cwd()
RESULTS = pathlib.Path(os.environ.get("P6_RESULTS", HERE.parent / "results")); RESULTS.mkdir(parents=True, exist_ok=True)
DATA = pathlib.Path(os.environ.get("P6_DATA", (RESULTS.parent / "data_quick") if QUICK else HERE.parent / "data")); DATA.mkdir(parents=True, exist_ok=True)      # quick mode never touches the real manifest
SRC_POPE = pathlib.Path(os.environ.get("P6_SRC", "C:/data/p6")) / "POPE" / "Full" / "adversarial-00000-of-00001.parquet"
SRC_VQA = sorted((pathlib.Path(os.environ.get("P6_SRC", "C:/data/p6")) / "VQAv2" / "data").glob("validation-*.parquet"))
INTERNVL_PATH = os.environ.get("P6_INTERNVL", "C:/models/InternVL3-8B-hf")

N_POPE, N_VQA_PER_TYPE, N_PARAPHRASE = (300, 100, 100) if not QUICK else (8, 3, 6)
SEED = int(os.environ.get("P6_SEED", 0))     # selects the sampled questions (replicate runs)
QWEN = "qwen2.5vl:7b"; QWEN3 = "qwen3-vl:8b-instruct"
A_NAME, B_NAME, REF_NAME = "qwen3-vl-8b-instruct (ollama q4)", "internvl3-8b (bnb nf4)", "qwen2.5-vl-7b (ollama q4)"     # system A, system B, reference
MAX_TILES = 6   # InternVL dynamic tiling cap (default 12). 12 tiles = about 3,300 image tokens and overflowed 8 GB (60 to 170 s per answer); 6 tiles = about 1,800 tokens
GEN_OPTS = {"temperature": 0, "seed": 0, "num_ctx": 4096, "num_predict": 16}
ELECTRICITY_AUD_PER_KWH = 0.30   # ASSUMPTION for the cost estimate, not measured
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("quick:", QUICK, "| POPE n:", N_POPE, "| VQA per type:", N_VQA_PER_TYPE, "| results:", RESULTS)

## 1. Environment record

In [ ]:
def smi(query):
    try: return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"], text=True).strip()
    except Exception as e: return f"unavailable ({e})"

models_info = {}
try:
    for m in ollama.list().models: models_info[m.model] = {"digest": (m.digest or "")[:12], "size_gb": round((m.size or 0) / 2**30, 2)}
except Exception as e: models_info = {"error": str(e)}
env_record = {"date": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "os": platform.platform(),
    "torch": torch.__version__, "transformers": im.version("transformers"), "ollama_client": im.version("ollama"), "pyarrow": im.version("pyarrow"),
    "gpu": smi("name"), "gpu_memory_total_mib": smi("memory.total"), "driver": smi("driver_version"),
    "qwen_model": {QWEN: models_info.get(QWEN)}, "qwen3_model": {QWEN3: models_info.get(QWEN3)}, "internvl_path": INTERNVL_PATH, "gen_options": GEN_OPTS,
    "n_pope": N_POPE, "n_vqa_per_type": N_VQA_PER_TYPE, "n_paraphrase": N_PARAPHRASE, "seed": SEED}
(RESULTS / "environment.json").write_text(json.dumps(env_record, indent=2)); print(json.dumps(env_record, indent=2))

## 2. Evaluation sets
- **POPE adversarial** (Li et al., 2023; COCO val2014 images): balanced yes/no questions about object presence. `N_POPE` questions are drawn at random (half answer yes, half no).
- **VQAv2 validation** (Goyal et al., 2017; CC BY 4.0): questions are kept when they mention a home or furniture word. Up to `N_VQA_PER_TYPE` questions are sampled per answer type.
- Images are saved as JPEG files and a manifest records every question, so the set can be rebuilt identically.

In [ ]:
HOME_WORDS = ["chair", "couch", "sofa", "bed", "table", "desk", "lamp", "shelf", "cabinet", "bench", "rug", "curtain", "pillow", "tv", "television",
              "clock", "vase", "kitchen", "living room", "bedroom", "furniture", "stool", "drawer", "mirror", "window", "door", "plant", "refrigerator", "oven", "sink", "bookshelf"]
IMG_DIR = DATA / "images"; IMG_DIR.mkdir(exist_ok=True)
MANIFEST = DATA / "eval_manifest.json"

def save_img(b, name):
    p = IMG_DIR / f"{name}.jpg"
    if not p.exists(): Image.open(io.BytesIO(b)).convert("RGB").save(p, quality=95)
    return str(p)

def build_sets():
    rng = np.random.RandomState(SEED); items = []
    pope = pq.read_table(SRC_POPE).to_pandas()
    for ans in ("yes", "no"):
        sub = pope[pope.answer.str.lower() == ans]
        for _, r in sub.sample(n=N_POPE // 2, random_state=SEED).iterrows():
            items.append({"set": "pope", "id": f"pope_{r.id}", "question": r.question, "truth": ans, "answers": [ans], "answer_type": "yes/no",
                          "image": save_img(r.image["bytes"], f"pope_{r.id}"), "source": r.image_source})
    vqa = pd.concat([pq.read_table(p).to_pandas() for p in SRC_VQA], ignore_index=True)
    mask = vqa.question.str.lower().apply(lambda q: any(w in q for w in HOME_WORDS))
    vqa = vqa[mask]
    for at in ("yes/no", "number", "other"):
        sub = vqa[vqa.answer_type == at]
        for _, r in sub.sample(n=min(N_VQA_PER_TYPE, len(sub)), random_state=SEED).iterrows():
            items.append({"set": "vqa", "id": f"vqa_{r.question_id}", "question": r.question, "truth": r.multiple_choice_answer,
                          "answers": [a["answer"] for a in r.answers], "answer_type": at,
                          "image": save_img(r.image["bytes"], f"vqa_{r.question_id}"), "source": f"COCO image {r.image_id}"})
    return items

if MANIFEST.exists() and not QUICK:
    ITEMS = json.loads(MANIFEST.read_text())
else:
    ITEMS = build_sets()
    MANIFEST.write_text(json.dumps(ITEMS, indent=1))
man = pd.DataFrame(ITEMS)
print(man.groupby(["set", "answer_type"]).size())

## 3. Models and prompts
Both models receive the same prompt text. Greedy decoding with at most 16 new tokens.

In [ ]:
PROMPT = {"pope": "{q} Answer with Yes or No.", "vqa": "{q} Answer the question using a single word or phrase."}
PARAPHRASE = "Looking at this picture, can you see {obj}? Answer with Yes or No."

class QwenOllama:
    name = "qwen2.5-vl-7b (ollama q4)"; model = QWEN; extra = {}; prefill = None
    def load(self): pass
    def ask(self, image_path, prompt):
        b = pathlib.Path(image_path).read_bytes()
        t = time.perf_counter()
        msgs = [{"role": "user", "content": prompt, "images": [b]}] + ([{"role": "assistant", "content": self.prefill}] if self.prefill else [])
        r = ollama.chat(model=self.model, messages=msgs, options=GEN_OPTS, keep_alive="30m", **self.extra)
        return r["message"]["content"].strip(), time.perf_counter() - t, int(r.get("prompt_eval_count", 0) or 0), int(r.get("eval_count", 0) or 0)
    def unload(self):
        try: ollama.generate(model=self.model, prompt="", keep_alive=0)
        except Exception: pass
        for _ in range(90):                                  # wait until Ollama has really released the model
            try:
                if not ollama.ps().models: break
            except Exception: break
            time.sleep(1)

class InternVL:
    name = "internvl3-8b (bnb nf4)"
    def load(self):
        from transformers import AutoProcessor, AutoModelForImageTextToText
        self.proc = AutoProcessor.from_pretrained(INTERNVL_PATH)
        from transformers import BitsAndBytesConfig
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
                               llm_int8_skip_modules=["vision_tower", "multi_modal_projector"])   # vision tower and projector stay in bf16; the output layer is quantised too to leave GPU headroom
        self.model = AutoModelForImageTextToText.from_pretrained(INTERNVL_PATH, quantization_config=q, dtype=torch.bfloat16, device_map="cuda").eval()
    def ask(self, image_path, prompt):
        img = Image.open(image_path).convert("RGB")
        msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
        t = time.perf_counter()
        inputs = self.proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", images_kwargs={"max_patches": MAX_TILES}).to(self.model.device, dtype=torch.bfloat16)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=16, do_sample=False)
        text = self.proc.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        torch.cuda.synchronize()
        return text, time.perf_counter() - t, int(inputs["input_ids"].shape[1]), int(out.shape[1] - inputs["input_ids"].shape[1])
    def unload(self):
        import gc; self.model = None; gc.collect(); torch.cuda.empty_cache()

class Qwen3Ollama(QwenOllama):
    name = "qwen3-vl-8b-instruct (ollama q4)"; model = QWEN3; extra = {}; prefill = None      # Qwen3-VL reasons before answering; an empty reasoning block is pre-filled (the documented way to switch reasoning off) so that every system answers directly

MODELS = [QwenOllama(), Qwen3Ollama(), InternVL()]

## 4. Inference run
Every answer is appended to `results/predictions.jsonl`. The run can be resumed. Each model is loaded alone to keep VRAM use separate.

In [ ]:
PRED = RESULTS / "predictions.jsonl"; ENERGY = RESULTS / "energy.jsonl"

class GpuSampler:
    def __init__(self): self.vals, self._run = [], False
    def _loop(self):
        while self._run:
            try:
                p, u, m = smi("power.draw,utilization.gpu,memory.used").split(", "); self.vals.append((float(p), float(u), float(m)))
            except Exception: pass
            time.sleep(1.0)
    def start(self): self._run = True; self.t0 = time.time(); self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()
    def stop(self):
        self._run = False; self.th.join(timeout=3); dur = time.time() - self.t0; a = np.array(self.vals) if self.vals else np.zeros((1, 3))
        return {"duration_s": round(dur, 1), "mean_power_w": round(float(a[:, 0].mean()), 1), "mean_util_pct": round(float(a[:, 1].mean()), 1),
                "peak_mem_mib": float(a[:, 2].max()), "energy_wh": round(float(a[:, 0].mean()) * dur / 3600, 3)}

def done_keys():
    if not PRED.exists(): return set()
    return {(r["model"], r["id"], r["variant"]) for r in map(json.loads, PRED.read_text().splitlines()) if r}

def paraphrase_items():
    out = []
    pope = [it for it in ITEMS if it["set"] == "pope"]
    rng = random.Random(SEED); rng.shuffle(pope)
    for it in pope:
        m = re.match(r"Is there (an? .+?) in the image\?", it["question"])
        if m: out.append((it, PARAPHRASE.format(obj=m.group(1))))
        if len(out) >= N_PARAPHRASE: break
    return out

def wait_gpu_free(limit_mib=2000, timeout=120):
    # a model that does not fit next to leftovers spills into shared memory and runs about 50 times slower
    for _ in range(timeout):
        try:
            if float(smi("memory.used").split("\n")[0]) < limit_mib: return
        except Exception: return
        time.sleep(1)
    print("warning: GPU memory still above", limit_mib, "MiB after", timeout, "s")

def run_model(model):
    done = done_keys()
    jobs = [(it, "original", PROMPT[it["set"]].format(q=it["question"])) for it in ITEMS]
    jobs += [(it, "paraphrase", p) for it, p in paraphrase_items()]
    todo = [j for j in jobs if (model.name, j[0]["id"], j[1]) not in done]
    if not todo: print(model.name, "already complete"); return
    base_mem = float(smi("memory.used").split("\n")[0]) if smi("memory.used")[0].isdigit() else 0.0
    torch.cuda.reset_peak_memory_stats()
    wait_gpu_free()
    t_load = time.time(); model.load(); load_s = time.time() - t_load
    sampler = GpuSampler(); sampler.start(); t0 = time.time()
    try:
        for i, (it, variant, prompt) in enumerate(todo):
            text, dt, ptok, ctok = model.ask(it["image"], prompt)
            rec = {"model": model.name, "id": it["id"], "set": it["set"], "variant": variant, "answer_type": it["answer_type"], "question": it["question"],
                   "prediction": text, "latency_s": round(dt, 3), "prompt_tokens": ptok, "completion_tokens": ctok}
            with PRED.open("a") as f: f.write(json.dumps(rec) + "\n")
            if i % 50 == 0: print(f"{model.name}: {i + 1}/{len(todo)} | last: '{text}' ({dt:.1f}s)", flush=True)
    finally:
        e = sampler.stop(); e.update({"model": model.name, "answers": len(todo), "load_s": round(load_s, 1), "wall_s": round(time.time() - t0, 1),
                                      "baseline_mem_mib": base_mem, "torch_peak_alloc_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)})
        with ENERGY.open("a") as f: f.write(json.dumps(e) + "\n")
        print("energy:", e)
    model.unload()

for m in MODELS: run_model(m)

## 5. Scoring
Answer normalisation: lower case, punctuation and articles removed, number words mapped to digits. VQA accuracy is `min(1, matches / 3)`, where matches is the number of the ten human answers equal to the prediction (a simplified form of the official metric). POPE answers are mapped to yes or no from the first word; anything else counts as wrong.

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
NUM = {"zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10"}
def norm(s):
    s = str(s).lower().strip().rstrip(".")
    s = s.translate(str.maketrans("", "", string.punctuation))
    toks = [NUM.get(w, w) for w in s.split() if w not in ("a", "an", "the")]
    return " ".join(toks)
def yn(s):
    w = norm(s).split()
    return w[0] if w and w[0] in ("yes", "no") else "invalid"

pred = pd.DataFrame([json.loads(l) for l in PRED.read_text().splitlines()])
items = {it["id"]: it for it in ITEMS}
pred["truth"] = pred.id.map(lambda i: items[i]["truth"])
def vqa_acc(r):
    p = norm(r.prediction); humans = [norm(a) for a in items[r.id]["answers"]]
    return min(1.0, sum(h == p for h in humans) / 3.0)
pope = pred[(pred.set == "pope")].copy(); pope["pred_yn"] = pope.prediction.map(yn); pope["correct"] = pope.pred_yn == pope.truth
import re
def lenient_acc(r):
    # lenient scoring for models that answer in full sentences: yes/no from the first word, numbers from the first digit token, other answers by whole-word match of the human answers
    pred_ = norm(r.prediction); humans = [norm(a) for a in items[r.id]["answers"]]; toks = pred_.split()
    if r.answer_type == "yes/no": q_ = toks[0] if toks and toks[0] in ("yes", "no") else ""; return min(1.0, sum(h == q_ for h in humans) / 3.0)
    if r.answer_type == "number": q_ = next((t for t in toks if t.isdigit()), pred_); return min(1.0, sum(h == q_ for h in humans) / 3.0)
    n_ = sum(1 for h in humans if h and re.search(r"(?<!\w)" + re.escape(h) + r"(?!\w)", pred_)); return min(1.0, n_ / 3.0)
vqa = pred[(pred.set == "vqa")].copy(); vqa["score"] = vqa.apply(vqa_acc, axis=1); vqa["score_lenient"] = vqa.apply(lenient_acc, axis=1)
pred["words"] = pred.prediction.astype(str).str.split().str.len()
pred.groupby("model").agg(answers=("id", "size"), mean_words=("words", "mean"), share_over_5_words=("words", lambda x: (x > 5).mean()), empty=("prediction", lambda x: (x.astype(str).str.strip() == "").sum())).round(3).to_csv(RESULTS / "answer_length.csv")
orig = pope[pope.variant == "original"]
print({m: len(g) for m, g in pred.groupby("model")})

In [ ]:
rows = []
for m, g in orig.groupby("model"):
    tp = ((g.pred_yn == "yes") & (g.truth == "yes")).sum(); fp = ((g.pred_yn == "yes") & (g.truth == "no")).sum()
    fn = ((g.pred_yn != "yes") & (g.truth == "yes")).sum(); tn = ((g.pred_yn == "no") & (g.truth == "no")).sum()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    rows.append({"model": m, "n": len(g), "accuracy": g.correct.mean(), "precision": prec, "recall": rec, "f1": 2 * prec * rec / max(prec + rec, 1e-9),
                 "yes_ratio": (g.pred_yn == "yes").mean(), "hallucination_rate": fp / max((g.truth == "no").sum(), 1), "invalid_answers": int((g.pred_yn == "invalid").sum())})
pope_tbl = pd.DataFrame(rows).round(3); pope_tbl.to_csv(RESULTS / "pope_summary.csv", index=False); pope_tbl

In [ ]:
vo = vqa[vqa.variant == "original"]
vqa_tbl = vo.pivot_table(index="answer_type", columns="model", values="score", aggfunc="mean").round(3)
vqa_tbl.loc["overall"] = vo.groupby("model").score.mean().round(3)
vqa_tbl.to_csv(RESULTS / "vqa_summary.csv")
vqa_len = vo.pivot_table(index="answer_type", columns="model", values="score_lenient", aggfunc="mean").round(3); vqa_len.loc["overall"] = vo.groupby("model").score_lenient.mean().round(3)
vqa_len.to_csv(RESULTS / "vqa_summary_lenient.csv")
print("strict VQA accuracy (exact match, as in the standard metric):"); display(vqa_tbl); print("lenient VQA accuracy (for answers given as full sentences, see the scoring text):"); display(vqa_len)

In [ ]:
# paired comparisons on identical questions: system A against system B, the reference against system B, system A against the reference
for a_, b_ in ((A_NAME, B_NAME), (REF_NAME, B_NAME), (A_NAME, REF_NAME)):
    if a_ not in set(pred.model) or b_ not in set(pred.model): continue
    w = orig.pivot_table(index="id", columns="model", values="correct", aggfunc="first").dropna()
    oa, ob = int(((w[a_] == True) & (w[b_] == False)).sum()), int(((w[a_] == False) & (w[b_] == True)).sum())
    p = stats.binomtest(oa, oa + ob, 0.5).pvalue if oa + ob else 1.0
    print(f"POPE paired: n={len(w)} | only {a_} correct={oa} | only {b_} correct={ob} | exact McNemar p={p:.4f}")
    wv = vo.pivot_table(index="id", columns="model", values="score", aggfunc="first").dropna()
    d = (wv[a_] - wv[b_]).values; boot = [np.random.choice(d, len(d)).mean() for _ in range(2000)]
    nz = d[d != 0]; pw = stats.wilcoxon(nz).pvalue if len(nz) > 5 else np.nan
    print(f"VQA paired: n={len(wv)} | mean diff ({a_} minus {b_}) = {d.mean():.3f}, 95% bootstrap CI [{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}] | Wilcoxon p={pw:.4f}")
    wl = vo.pivot_table(index="id", columns="model", values="score_lenient", aggfunc="first").dropna(); dl = (wl[a_] - wl[b_]).values; bl = [np.random.choice(dl, len(dl)).mean() for _ in range(2000)]
    print(f"VQA lenient paired: mean diff ({a_} minus {b_}) = {dl.mean():.3f}, 95% bootstrap CI [{np.percentile(bl, 2.5):.3f}, {np.percentile(bl, 97.5):.3f}]")

In [ ]:
# robustness: answer flips when the same POPE question is reworded
pv = pope.pivot_table(index=["model", "id"], columns="variant", values="pred_yn", aggfunc="first").dropna()
flip = pv.groupby("model").apply(lambda g: pd.Series({"pairs": len(g), "flip_rate": (g["original"] != g["paraphrase"]).mean()})).round(3)
flip.to_csv(RESULTS / "robustness_flip_rate.csv"); flip

In [ ]:
# cost and speed
en = pd.DataFrame([json.loads(l) for l in ENERGY.read_text().splitlines()]).drop_duplicates("model", keep="last").set_index("model")
lat = pred.groupby("model").agg(mean_latency_s=("latency_s", "mean"), p95_latency_s=("latency_s", lambda x: np.percentile(x, 95)),
                                prompt_tokens=("prompt_tokens", "mean"), answers=("id", "size"))
biz = lat.join(en[["load_s", "mean_power_w", "peak_mem_mib", "baseline_mem_mib", "energy_wh", "torch_peak_alloc_gb"]])
biz["answers_per_hour"] = 3600 / biz.mean_latency_s
biz["gpu_wh_per_1000_answers"] = biz.energy_wh / biz.answers * 1000
biz["aud_per_1000_answers"] = biz.gpu_wh_per_1000_answers / 1000 * ELECTRICITY_AUD_PER_KWH
biz["extra_vram_mib"] = biz.peak_mem_mib - biz.baseline_mem_mib
biz = biz.round(3); biz.to_csv(RESULTS / "business.csv")
print("Electricity price is an assumption:", ELECTRICITY_AUD_PER_KWH, "AUD/kWh. Energy is GPU power only.")
biz

In [ ]:
# failure catalogue: hallucinations (said yes, object absent) and low-scoring VQA answers
h = orig[(orig.pred_yn == "yes") & (orig.truth == "no")][["model", "id", "question", "prediction"]]
l = vo[vo.score == 0][["model", "id", "question", "prediction", "answer_type"]]
h.to_csv(RESULTS / "failures_hallucination.csv", index=False); l.to_csv(RESULTS / "failures_vqa.csv", index=False)
print("hallucinations per model:\n", h.groupby("model").size().to_string(), "\nzero-score VQA answers per model:\n", l.groupby("model").size().to_string())
fig_dir = RESULTS / "figures"; fig_dir.mkdir(exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
vqa_tbl.plot.bar(ax=ax[0]); ax[0].set_title("VQA accuracy by answer type"); ax[0].set_ylim(0, 1); ax[0].tick_params(axis="x", rotation=0)
pope_tbl.set_index("model")[["accuracy", "f1", "yes_ratio", "hallucination_rate"]].plot.bar(ax=ax[1]); ax[1].set_title("POPE adversarial"); ax[1].set_ylim(0, 1); ax[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.savefig(fig_dir / "accuracy.png", dpi=150); plt.show()

## 6. Notes for the report
Interpretation, limits and recommendations are written after the full run. Limits already known: the two models differ in vision encoder, training data, 4-bit scheme (GGUF Q4 in llama.cpp vs NF4 in bitsandbytes, vision tower kept in higher precision in both) and runtime; POPE and VQAv2 use COCO photographs, not catalogue images; VQA accuracy uses the simplified form of the metric; sample sizes are 300 (POPE) and up to 300 (VQA); one prompt wording per task.